# Titanic Survival Prediction
This notebook demonstrates a complete machine learning workflow to predict passenger survival on the Titanic using Logistic Regression.

In [28]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder,OneHotEncoder,LabelEncoder,StandardScaler,MinMaxScaler

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer


from sklearn.svm import SVC

## 1. Data Loading and Initial Exploration
First, we load the dataset and take a look at a few sample rows to understand the features we are working with.

In [29]:
# Load the dataset from a CSV file into a pandas DataFrame
df = pd.read_csv("titanic_data_updated.csv")

# Display 5 random rows to see what the data looks like
df.sample(5)
#to_make csv from the original dataset, you can use the following code:


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
479,480,yes,third,"Hirvonen, Miss. Hildur E",female,2.0,0,1,3101298,12.2875,NaN,S
64,65,no,first,"Stewart, Mr. Albert A",male,NaN,0,0,PC 17605,27.7208,NaN,C
31,32,yes,first,"Spencer, Mrs. William Augustus (Marie Eugenie)",female,NaN,1,0,PC 17569,146.5208,B78,C
795,796,no,second,"Otter, Mr. Richard",male,39.0,0,0,28213,13.0000,NaN,S
735,736,no,third,"Williams, Mr. Leslie",male,28.5,0,0,54636,16.1000,NaN,S


In [30]:
df['Cabin'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 891 entries, 0 to 890
Series name: Cabin
Non-Null Count  Dtype 
--------------  ----- 
204 non-null    object
dtypes: object(1)
memory usage: 7.1+ KB


## 2. Feature Engineering
We create new features like `Family_Size` and extract the `Deck` from the `Cabin` column to provide more meaningful information to the model.

In [31]:
df['Family_Size'] = df['SibSp'] + df['Parch'] + 1
df['Cabin'] = df['Cabin'].fillna("Missing")

df['Deck'] = df['Cabin'].astype(str).str[0]
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_Size,Deck
611,612,no,third,"Jardin, Mr. Jose Neto",male,NaN,0,0,SOTON/O.Q. 3101305,7.05,Missing,S,1,M
217,218,no,second,"Jacobsohn, Mr. Sidney Samuel",male,42.0,1,0,243847,27.00,Missing,S,2,M
734,735,no,second,"Troupiansky, Mr. Moses Aaron",male,23.0,0,0,233639,13.00,Missing,S,1,M
456,457,no,first,"Millet, Mr. Francis Davis",male,65.0,0,0,13509,26.55,E38,S,1,E
33,34,no,second,"Wheadon, Mr. Edward H",male,66.0,0,0,C.A. 24579,10.50,Missing,S,1,M


In [32]:
df['Deck'].value_counts()

Deck
M    687
C     59
B     47
D     33
E     32
A     15
F     13
G      4
T      1
Name: count, dtype: int64

In [33]:
# Separate the features (X) from the target we want to predict (y)
# We drop 'Survived' from X because that's our answer key
X = df.drop('Survived', axis=1)

# y contains only the survival status
y = df['Survived']

## 3. Data Splitting
We split the data into training (to teach the model) and testing (to evaluate the model) sets. This ensures we can test the model on data it hasn't seen before.

In [34]:
# Split data: 80% for training the model and 20% for testing its accuracy
# 'stratify=y' ensures both sets have a similar percentage of survivors
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## 4. Outlier Handling and Data Cleaning
We use Z-scores to remove extreme outliers in `Age` and 'clip' the `Fare` column to prevent extreme values from distorting our model's learning.

In [35]:
X_train

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_Size,Deck
692,693,third,"Lam, Mr. Ali",male,NaN,0,0,1601,56.4958,Missing,S,1,M
481,482,second,"Frost, Mr. Anthony Wood ""Archie""",male,NaN,0,0,239854,0.0000,Missing,S,1,M
527,528,first,"Farthing, Mr. John",male,NaN,0,0,PC 17483,221.7792,C95,S,1,C
855,856,third,"Aks, Mrs. Sam (Leah Rosen)",female,18.0,0,1,392091,9.3500,Missing,S,2,M
801,802,second,"Collyer, Mrs. Harvey (Charlotte Annie Tate)",female,31.0,1,1,C.A. 31921,26.2500,Missing,S,3,M
...,...,...,...,...,...,...,...,...,...,...,...,...,...
359,360,third,"Mockler, Miss. Helen Mary ""Ellie""",female,NaN,0,0,330980,7.8792,Missing,Q,1,M
258,259,first,"Ward, Miss. Anna",female,35.0,0,0,PC 17755,512.3292,Missing,C,1,M
736,737,third,"Ford, Mrs. Edward (Margaret Ann Watson)",female,48.0,1,3,W./C. 6608,34.3750,Missing,S,5,M
462,463,first,"Gee, Mr. Arthur H",male,47.0,0,0,111320,38.5000,E63,S,1,E


In [36]:
y_train

692    yes
481     no
527     no
855    yes
801    yes
      ... 
359    yes
258    yes
736     no
462     no
507    yes
Name: Survived, Length: 712, dtype: object

In [37]:
#age
mean_age = X_train['Age'].mean()
std_age = X_train['Age'].std()

X_train['Z_score'] = (X_train['Age'] - mean_age) / std_age

musk = (abs(X_train['Z_score']) <= 3)

X_train = X_train[musk]
y_train = y_train[musk]

# fare

fare_Q1 = X_train['Fare'].quantile(0.25)
fare_Q3 = X_train['Fare'].quantile(0.75)

IQR = fare_Q3 - fare_Q1

minimum = max(0 , fare_Q1 - 1.5 * IQR)
maximum = fare_Q3 + 1.5 * IQR

X_train['Fare'] = X_train['Fare'].clip(minimum, maximum)

C:\Users\user\AppData\Local\Temp\ipykernel_13876\987313337.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train['Fare'] = X_train['Fare'].clip(minimum, maximum)


## 5. Building Preprocessing Pipelines
We create different pipelines for numerical and categorical data. This handles missing values (imputation) and scales the data so all features are on a similar range.

In [38]:
# pipeline

# numerical
p1 = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='mean')),
        ('scaler',StandardScaler())
    ]
)

p2 = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='median')),
        ('scaler',MinMaxScaler())
    ]
)

In [39]:
# categories goes one after another via stricly following the order of the input
categories = [['third','second','first']]

In [40]:
# pipeline

#  categorical columns

p3 = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='most_frequent')),
        ('encoder',OneHotEncoder(sparse_output=False,drop='first',handle_unknown='ignore'))
    ]
)

p4 = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='most_frequent')),
        ('encoder',OrdinalEncoder(categories=categories)),
        ('scaler',MinMaxScaler())
    ]
)

In [41]:
preprocessor = ColumnTransformer(
    transformers=[
        ('pipeline_1',p1,['Age']),
        ('pipeline_2',p2,['Fare','Family_Size']),
        ('pipeline_3',p3,['Embarked','Sex','Deck']),
        ('pipeline_4',p4,['Pclass'])
    ],
    remainder='drop'
)
preprocessor

,transformers,"[('pipeline_1', ...), ('pipeline_2', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'mean'
,fill_value,None


## 6. Label Encoding
We convert our target variable (`yes`/`no`) into numerical values (1/0) because machine learning models require numeric inputs.

In [42]:
le = LabelEncoder()

le.fit(y_train)

y_train = le.transform(y_train)
y_test = le.transform(y_test)



## 7. Model Training
We combine the preprocessing and the Logistic Regression model into a single `lr_model` pipeline and fit it to our training data.

In [43]:
SVC_model = Pipeline(
    steps=[
        ('preprocessor',preprocessor),
        ('model',SVC())
    ]
)

In [44]:
from sklearn.model_selection import GridSearchCV

In [45]:
grid_param =[ {
    "model__kernel" : ['linear'],
    "model__C" :[0.01 , 0.1 , 1 , 10 , 50 , 100]
},
  {
      "model__kernel" : ['rbf'] ,
      "model__C" :[0.01 , 0.1 ,1,100],
      "model__gamma" :[0.01,0.1,5,10,'scale','auto']
  },
    {
        "model__kernel": ['poly'],
        "model__C" :[0.01 , 0.1 , 1 ,100],
        "model__degree" : [2,3]
    }

]



In [46]:
best_SVC_model = GridSearchCV(estimator = SVC_model , param_grid = grid_param , cv = 5)

In [47]:
best_SVC_model.fit(X_train,y_train)

C:\Users\user\AppData\Roaming\Python\Python312\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
C:\Users\user\AppData\Roaming\Python\Python312\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
C:\Users\user\AppData\Roaming\Python\Python312\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
C:\Users\user\AppData\Roaming\Python\Python312\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
C:\Users\user\AppData\Roamin

,estimator,"Pipeline(step...del', SVC())])"
,param_grid,"[{'model__C': [0.01, 0.1, ...], 'model__kernel': ['linear']}, {'model__C': [0.01, 0.1, ...], 'model__gamma': [0.01, 0.1, ...], 'model__kernel': ['rbf']}, ...]"
,scoring,None
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('pipeline_1', ...), ('pipeline_2', ...), ...]"


In [52]:
# training accuracy
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,confusion_matrix
y_pred_train = best_SVC_model.predict(X_train)

print(accuracy_score(y_train,y_pred_train))

0.8429319371727748


In [53]:
y_pred = best_SVC_model.predict(X_test)

accuracy = accuracy_score(y_test,y_pred)
print(accuracy)
precision = precision_score(y_test,y_pred)
print(precision)
recall = recall_score(y_test,y_pred)
print(recall)

0.8044692737430168
0.8148148148148148
0.6376811594202898
